# MODIS True Color Animation for Lake Monitoring

This notebook creates time-series animations showing daily MODIS imagery (Terra + Aqua combined) for Detroit Lake and Upper Klamath Lake from 2011-present, with labels indicating the masking status for the specified Region of Interest (ROI).

## Features
- Combines MODIS Terra (morning) and Aqua (afternoon) imagery
- 60km × 120km context area with 500m × 500m ROI overlay
- Uses 250m resolution bands for RGB with 500m band subsampling
- True color processing following MODIS best practices
- Cloud/snow/land masking detection with color-coded status badges
- Exports as GIF and MP4 formats

In [ ]:
# Import required libraries
import ee
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import animation
from matplotlib.patches import Rectangle
import pandas as pd
from datetime import datetime, timedelta, timezone
import os
import imageio
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Initialize Google Earth Engine
ee.Authenticate()
ee.Initialize(project='ee-toddsteissberg')

print("Google Earth Engine initialized successfully!")

In [ ]:
# Configuration
LAKES = [
    {
        'name': 'Detroit Lake',
        'lon': -122.184,
        'lat': 44.711,
        'export_prefix': 'detroit'
    },
    {
        'name': 'Upper Klamath Lake',
        'lon': -121.900,
        'lat': 42.400,
        'export_prefix': 'klamath'
    }
]

# Date range
START_DATE = '2011-01-01'
END_DATE = '2024-12-31'

# Spatial parameters
ROI_SIZE = 250  # 500m total (250m buffer)
CONTEXT_NS = 30000  # 60km total (30km buffer North-South)
CONTEXT_EW = 60000  # 120km total (60km buffer East-West)

# Animation parameters
FRAME_RATE = 10  # frames per second
FIGURE_SIZE = (12, 12)  # inches
DPI = 100  # dots per inch

# Output directory
OUTPUT_DIR = 'modis_animations'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Data Extraction Functions

In [ ]:
def get_modis_collection(lake_info, start_date, end_date):
    """
    Get combined MODIS Terra and Aqua collection for a lake.
    """
    # Define center point and regions
    center = ee.Geometry.Point([lake_info['lon'], lake_info['lat']])
    roi = center.buffer(ROI_SIZE).bounds()
    
    # Create rectangular context region (60km NS x 120km EW)
    # Calculate coordinates for rectangle
    lat_buffer = CONTEXT_NS / 111000  # Approximate degrees (111 km per degree latitude)
    lon_buffer = CONTEXT_EW / (111000 * ee.Number(center.coordinates().get(1)).cos().getInfo())  # Adjust for longitude
    
    context = ee.Geometry.Rectangle([
        lake_info['lon'] - lon_buffer,  # west
        lake_info['lat'] - lat_buffer,  # south  
        lake_info['lon'] + lon_buffer,  # east
        lake_info['lat'] + lat_buffer   # north
    ])
    
    # Get Terra collection
    terra = ee.ImageCollection('MODIS/061/MOD09GA') \
        .filterDate(start_date, end_date) \
        .filterBounds(context) \
        .map(lambda img: img.set('satellite', 'Terra'))
    
    # Get Aqua collection
    aqua = ee.ImageCollection('MODIS/061/MYD09GA') \
        .filterDate(start_date, end_date) \
        .filterBounds(context) \
        .map(lambda img: img.set('satellite', 'Aqua'))
    
    # Merge collections
    combined = terra.merge(aqua).sort('system:time_start')
    
    return combined, roi, context


def check_roi_mask_status(image, roi):
    """
    Check masking status within ROI (clouds, snow, land).
    """
    try:
        qa = image.select('state_1km')
        cloud_mask = qa.bitwiseAnd(1 << 10).gt(0)
        cirrus_mask = qa.bitwiseAnd(3 << 8).gt(0)
        snow_mask = qa.bitwiseAnd(1 << 12).gt(0)
        bad_pixels = cloud_mask.Or(cirrus_mask).Or(snow_mask)
        
        stats = bad_pixels.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=roi,
            scale=500,
            maxPixels=1e9
        )
        
        masked_fraction = ee.Number(stats.get('state_1km')).getInfo()
        
        if masked_fraction < 0.1:
            return 'clear'
        elif masked_fraction > 0.9:
            return 'masked'
        else:
            return 'partial'
    except:
        return 'unknown'


def check_chlorophyll_algorithm_mask(image, roi):
    """
    Check if this image would be excluded from chlorophyll time series.
    """
    try:
        qa = image.select('state_1km')
        cloud = qa.bitwiseAnd(1 << 10).gt(0)
        snow = qa.bitwiseAnd(1 << 12).gt(0)
        cloud_snow_mask = cloud.Or(snow)
        
        mask_stats = cloud_snow_mask.reduceRegion(
            reducer=ee.Reducer.max(),
            geometry=roi,
            scale=500,
            maxPixels=1e9
        )
        
        has_cloud_snow = ee.Number(mask_stats.get('state_1km')).getInfo()
        return has_cloud_snow > 0
        
    except:
        return True


def get_true_color_image(image, context):
    """
    Extract true color RGB values from MODIS image.
    """
    try:
        # Select bands and scale
        rgb = image.select(['sur_refl_b01', 'sur_refl_b04', 'sur_refl_b03']).multiply(0.0001).clip(context)
        
        # Use sampleRectangle WITHOUT scale parameter - it uses native resolution
        rgb_data = rgb.sampleRectangle(
            region=context,
            defaultValue=0
        )
        
        # Extract arrays
        red_arr = np.array(rgb_data.get('sur_refl_b01').getInfo())
        green_arr = np.array(rgb_data.get('sur_refl_b04').getInfo())
        blue_arr = np.array(rgb_data.get('sur_refl_b03').getInfo())
        
        # Check if arrays are valid
        if red_arr.size == 0 or green_arr.size == 0 or blue_arr.size == 0:
            # Return default size array for 60km x 120km at 500m = 120x240 pixels
            return np.zeros((120, 240, 3))
        
        # Stack into RGB
        rgb_image = np.dstack([red_arr, green_arr, blue_arr])
        
        # Simple enhancement
        rgb_enhanced = np.zeros_like(rgb_image)
        for i in range(3):
            band = rgb_image[:,:,i]
            if np.max(band) > 0:
                rgb_enhanced[:,:,i] = np.clip(band * 3.0, 0, 1)
        
        return rgb_enhanced
        
    except Exception as e:
        print(f"Error in get_true_color_image: {e}")
        return np.zeros((120, 240, 3))

## Visualization Functions

In [ ]:
def create_frame(image_data, lake_info, roi_bounds, datetime_obj, satellite, mask_status, chl_excluded):
    """
    Create a single animation frame with all overlays.
    """
    # Adjust figure size for rectangular aspect ratio (2:1)
    fig, ax = plt.subplots(figsize=(16, 8), dpi=DPI)
    
    # Display the true color image with correct extent
    extent_ns_km = CONTEXT_NS / 1000  # 30 km
    extent_ew_km = CONTEXT_EW / 1000  # 60 km
    ax.imshow(image_data, extent=[-extent_ew_km, extent_ew_km, -extent_ns_km, extent_ns_km])
    
    # Add ROI box (gold, transparent)
    roi_km = ROI_SIZE / 1000
    roi_rect = Rectangle(
        xy=(-roi_km, -roi_km),
        width=roi_km * 2,
        height=roi_km * 2,
        linewidth=2,
        edgecolor='gold',
        facecolor='none',
        alpha=0.8
    )
    ax.add_patch(roi_rect)
    
    # Add title
    ax.set_title(f"{lake_info['name']}", fontsize=16, fontweight='bold', pad=20)
    
    # Format datetime with hour and minute
    date_str = datetime_obj.strftime('%Y-%m-%d %H:%M UTC')
    date_label = f"{date_str} | MODIS-{satellite}"
    ax.text(0.02, 0.98, date_label, transform=ax.transAxes,
            fontsize=12, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Add mask status badge
    badge_colors = {
        'clear': 'green',
        'masked': 'red',
        'partial': 'gold',
        'unknown': 'gray'
    }
    badge_text = f"ROI: {mask_status.upper()}"
    ax.text(0.98, 0.98, badge_text, transform=ax.transAxes,
            fontsize=12, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor=badge_colors.get(mask_status, 'gray'), 
                     alpha=0.7, edgecolor='black', linewidth=1))
    
    # Add chlorophyll algorithm exclusion indicator
    chl_text = "Chl Algorithm: EXCLUDED" if chl_excluded else "Chl Algorithm: INCLUDED"
    chl_color = 'red' if chl_excluded else 'lightgreen'
    ax.text(0.98, 0.92, chl_text, transform=ax.transAxes,
            fontsize=11, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor=chl_color, 
                     alpha=0.7, edgecolor='black', linewidth=1))
    
    # Add scale bar (20 km for the larger image)
    scale_length = 20  # km
    scale_x = extent_ew_km * 0.6
    scale_y = -extent_ns_km * 0.8
    ax.plot([scale_x, scale_x + scale_length], [scale_y, scale_y], 'k-', linewidth=3)
    ax.text(scale_x + scale_length/2, scale_y + 2, f'{scale_length} km', 
            ha='center', fontsize=10, backgroundcolor='white')
    
    # Set axis labels and limits
    ax.set_xlabel('Distance (km)', fontsize=12)
    ax.set_ylabel('Distance (km)', fontsize=12)
    ax.set_xlim(-extent_ew_km, extent_ew_km)
    ax.set_ylim(-extent_ns_km, extent_ns_km)
    ax.grid(True, alpha=0.3)
    
    return fig


def process_image_batch(images, lake_info, roi, context, batch_name):
    """
    Process a batch of images and save frames.
    Includes EVERY image in the time period.
    """
    frames_dir = os.path.join(OUTPUT_DIR, f"{lake_info['export_prefix']}_frames")
    os.makedirs(frames_dir, exist_ok=True)
    
    frame_paths = []
    image_list = images.getInfo()['features']
    
    print(f"Processing ALL {len(image_list)} images for {lake_info['name']} - {batch_name}")
    
    for i, img_info in enumerate(tqdm(image_list)):
        try:
            # Get image
            image = ee.Image(img_info['id'])
            
            # Get metadata with full datetime
            timestamp_ms = img_info['properties']['system:time_start']
            datetime_obj = datetime.fromtimestamp(timestamp_ms/1000, tz=timezone.utc)
            
            # Get satellite
            satellite = img_info['properties'].get('satellite', 'Unknown')
            
            # Estimate actual overpass time based on satellite
            # MODIS Terra crosses equator at ~10:30 local solar time
            # MODIS Aqua crosses equator at ~13:30 local solar time
            # For US West Coast (UTC-8), this is roughly:
            if satellite == 'Terra':
                # Terra: ~18:30 UTC (10:30 PST)
                datetime_obj = datetime_obj.replace(hour=18, minute=30)
            elif satellite == 'Aqua':
                # Aqua: ~21:30 UTC (13:30 PST)
                datetime_obj = datetime_obj.replace(hour=21, minute=30)
            
            # Format filename as yyyy-mm-dd-hhmm (using estimated time)
            filename_date = datetime_obj.strftime('%Y-%m-%d-%H%M')
            
            # Check mask status
            mask_status = check_roi_mask_status(image, roi)
            
            # Check if excluded from chlorophyll algorithm
            chl_excluded = check_chlorophyll_algorithm_mask(image, roi)
            
            # Get true color image
            rgb_image = get_true_color_image(image, context)
            
            # Create frame
            fig = create_frame(rgb_image, lake_info, roi.getInfo(), 
                             datetime_obj, satellite, mask_status, chl_excluded)
            
            # Save frame with datetime in filename
            frame_path = os.path.join(frames_dir, f"frame_{filename_date}_{satellite}.png")
            fig.savefig(frame_path, bbox_inches='tight', dpi=DPI)
            plt.close(fig)
            
            frame_paths.append(frame_path)
            
        except Exception as e:
            print(f"Error processing image {i}: {e}")
            continue
    
    return frame_paths

In [ ]:
def create_frame(image_data, lake_info, roi_bounds, datetime_obj, satellite, mask_status, chl_excluded):
    """
    Create a single animation frame with all overlays.
    """
    fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
    
    # Display the true color image
    extent_km = CONTEXT_SIZE / 1000
    ax.imshow(image_data, extent=[-extent_km, extent_km, -extent_km, extent_km])
    
    # Add ROI box (gold, transparent)
    roi_km = ROI_SIZE / 1000
    roi_rect = Rectangle(
        xy=(-roi_km, -roi_km),
        width=roi_km * 2,
        height=roi_km * 2,
        linewidth=2,
        edgecolor='gold',
        facecolor='none',
        alpha=0.8
    )
    ax.add_patch(roi_rect)
    
    # Add title
    ax.set_title(f"{lake_info['name']}", fontsize=16, fontweight='bold', pad=20)
    
    # Format datetime with hour and minute
    date_str = datetime_obj.strftime('%Y-%m-%d %H:%M UTC')
    date_label = f"{date_str} | MODIS-{satellite}"
    ax.text(0.02, 0.98, date_label, transform=ax.transAxes,
            fontsize=12, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Add mask status badge
    badge_colors = {
        'clear': 'green',
        'masked': 'red',
        'partial': 'gold'
    }
    badge_text = f"ROI: {mask_status.upper()}"
    ax.text(0.98, 0.98, badge_text, transform=ax.transAxes,
            fontsize=12, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor=badge_colors[mask_status], 
                     alpha=0.7, edgecolor='black', linewidth=1))
    
    # Add chlorophyll algorithm exclusion indicator
    chl_text = "Chl Algorithm: EXCLUDED" if chl_excluded else "Chl Algorithm: INCLUDED"
    chl_color = 'red' if chl_excluded else 'lightgreen'
    ax.text(0.98, 0.92, chl_text, transform=ax.transAxes,
            fontsize=11, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor=chl_color, 
                     alpha=0.7, edgecolor='black', linewidth=1))
    
    # Add scale bar (10 km)
    scale_length = 10  # km
    scale_x = extent_km * 0.6
    scale_y = -extent_km * 0.9
    ax.plot([scale_x, scale_x + scale_length], [scale_y, scale_y], 'k-', linewidth=3)
    ax.text(scale_x + scale_length/2, scale_y + 1, f'{scale_length} km', 
            ha='center', fontsize=10, backgroundcolor='white')
    
    # Set axis labels and limits
    ax.set_xlabel('Distance (km)', fontsize=12)
    ax.set_ylabel('Distance (km)', fontsize=12)
    ax.set_xlim(-extent_km, extent_km)
    ax.set_ylim(-extent_km, extent_km)
    ax.grid(True, alpha=0.3)
    
    return fig


def process_image_batch(images, lake_info, roi, context, batch_name):
    """
    Process a batch of images and save frames.
    Includes EVERY image in the time period.
    """
    frames_dir = os.path.join(OUTPUT_DIR, f"{lake_info['export_prefix']}_frames")
    os.makedirs(frames_dir, exist_ok=True)
    
    frame_paths = []
    image_list = images.getInfo()['features']
    
    print(f"Processing ALL {len(image_list)} images for {lake_info['name']} - {batch_name}")
    
    for i, img_info in enumerate(tqdm(image_list)):
        try:
            # Get image
            image = ee.Image(img_info['id'])
            
            # Get metadata with full datetime
            timestamp_ms = img_info['properties']['system:time_start']
            datetime_obj = datetime.fromtimestamp(timestamp_ms/1000, tz=timezone.utc)
            
            # Format filename as yyyy-mm-dd-hhmm (using UTC time)
            filename_date = datetime_obj.strftime('%Y-%m-%d-%H%M')
            satellite = img_info['properties'].get('satellite', 'Unknown')
            
            # Check mask status
            mask_status = check_roi_mask_status(image, roi)
            
            # Check if excluded from chlorophyll algorithm
            chl_excluded = check_chlorophyll_algorithm_mask(image, roi)
            
            # Get true color image with proper processing
            rgb_image = get_true_color_image(image, context)
            
            # Create frame
            fig = create_frame(rgb_image, lake_info, roi.getInfo(), 
                             datetime_obj, satellite, mask_status, chl_excluded)
            
            # Save frame with datetime in filename
            frame_path = os.path.join(frames_dir, f"frame_{filename_date}_{satellite}.png")
            fig.savefig(frame_path, bbox_inches='tight', dpi=DPI)
            plt.close(fig)
            
            frame_paths.append(frame_path)
            
        except Exception as e:
            print(f"Error processing image {i}: {e}")
            # Still create a placeholder frame for missing data
            fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
            
            datetime_obj = datetime.fromtimestamp(img_info['properties']['system:time_start']/1000, 
                                                 tz=timezone.utc)
            date_str = datetime_obj.strftime('%Y-%m-%d %H:%M')
            
            ax.text(0.5, 0.5, f"Data unavailable\n{date_str}", 
                   transform=ax.transAxes, ha='center', va='center', fontsize=14)
            ax.set_title(f"{lake_info['name']}", fontsize=16, fontweight='bold')
            ax.set_xlim(-20, 20)
            ax.set_ylim(-20, 20)
            ax.set_xlabel('Distance (km)', fontsize=12)
            ax.set_ylabel('Distance (km)', fontsize=12)
            
            filename_date = datetime_obj.strftime('%Y-%m-%d-%H%M')
            frame_path = os.path.join(frames_dir, f"frame_{filename_date}_error.png")
            fig.savefig(frame_path, bbox_inches='tight', dpi=DPI)
            plt.close(fig)
            frame_paths.append(frame_path)
            continue
    
    return frame_paths

In [ ]:
def create_animation(frame_paths, output_path, fps=FRAME_RATE):
    """
    Create animation from frames.
    """
    # Sort frames by date and time
    frame_paths.sort()
    
    # Create GIF
    gif_path = output_path.replace('.mp4', '.gif')
    images = []
    for path in frame_paths:
        images.append(imageio.imread(path))
    
    imageio.mimsave(gif_path, images, fps=fps)
    print(f"GIF saved: {gif_path}")
    
    # Create MP4 (requires ffmpeg)
    try:
        writer = imageio.get_writer(output_path, fps=fps)
        for path in frame_paths:
            writer.append_data(imageio.imread(path))
        writer.close()
        print(f"MP4 saved: {output_path}")
    except:
        print("MP4 export requires ffmpeg. GIF was created successfully.")


def process_lake_yearly(lake_info, year):
    """
    Process one year of data for a lake.
    """
    start = f"{year}-01-01"
    end = f"{year}-12-31"
    
    # Get MODIS collection with 250m bands
    collection, roi, context = get_modis_collection(lake_info, start, end)
    
    # Process images
    batch_name = f"Year {year}"
    frame_paths = process_image_batch(collection, lake_info, roi, context, batch_name)
    
    return frame_paths

## Main Processing Loop

In [ ]:
# Process each lake
for lake in LAKES:
    print(f"\n{'='*50}")
    print(f"Processing {lake['name']}")
    print(f"{'='*50}")
    
    all_frames = []
    
    # Process year by year to manage memory
    for year in range(2011, 2025):
        print(f"\nProcessing year {year}...")
        try:
            frames = process_lake_yearly(lake, year)
            all_frames.extend(frames)
        except Exception as e:
            print(f"Error processing year {year}: {e}")
            continue
    
    # Create final animation
    if all_frames:
        output_path = os.path.join(OUTPUT_DIR, f"{lake['export_prefix']}_modis_animation.mp4")
        create_animation(all_frames, output_path)
        print(f"\nAnimation complete for {lake['name']}!")
        print(f"Total frames: {len(all_frames)}")
    else:
        print(f"No frames generated for {lake['name']}")

print("\n" + "="*50)
print("All animations completed!")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Test the functions
def test_everything():
    """Test to see what's working and what's not."""
    print("Testing MODIS data extraction with 60km x 120km extraction...")
    
    lake = LAKES[0]  # Detroit Lake
    print(f"Testing lake: {lake['name']} at {lake['lat']}, {lake['lon']}")
    
    # Test collection
    try:
        collection, roi, context = get_modis_collection(lake, '2024-01-01', '2024-01-05')
        image_list = collection.getInfo()['features']
        print(f"✓ Found {len(image_list)} images")
        
        if len(image_list) > 0:
            # Test first image
            img_info = image_list[0]
            print(f"✓ First image ID: {img_info['id']}")
            
            # Test timestamp
            timestamp_ms = img_info['properties']['system:time_start']
            dt = datetime.fromtimestamp(timestamp_ms/1000, tz=timezone.utc)
            print(f"✓ Timestamp: {dt} (ms: {timestamp_ms})")
            
            # Test satellite property
            satellite = img_info['properties'].get('satellite', 'NOT_FOUND')
            print(f"✓ Satellite: {satellite}")
            
            # Test context geometry bounds
            context_bounds = context.bounds().getInfo()
            print(f"✓ Context bounds: {context_bounds}")
            
            # Test image extraction
            image = ee.Image(img_info['id'])
            rgb_image = get_true_color_image(image, context)
            print(f"✓ Image extraction: {rgb_image.shape}, non-zero pixels: {np.count_nonzero(rgb_image)}")
            
            # Test masking
            mask_status = check_roi_mask_status(image, roi)
            chl_excluded = check_chlorophyll_algorithm_mask(image, roi)
            print(f"✓ Mask status: {mask_status}, Chl excluded: {chl_excluded}")
            
        else:
            print("✗ No images found")
            
    except Exception as e:
        print(f"✗ Error: {e}")
        import traceback
        traceback.print_exc()

# Run the test
test_everything()

In [ ]:
def create_preview(lake_info, num_days=30):
    """
    Create a quick preview animation with limited data.
    """
    # Use recent data for preview
    end_date = datetime.now()
    start_date = end_date - timedelta(days=num_days)
    
    print(f"Creating {num_days}-day preview for {lake_info['name']} with 60km x 120km extraction...")
    
    # Get MODIS collection
    collection, roi, context = get_modis_collection(
        lake_info, 
        start_date.strftime('%Y-%m-%d'),
        end_date.strftime('%Y-%m-%d')
    )
    
    # Process images
    frames = process_image_batch(collection, lake_info, roi, context, "Preview")
    
    # Create animation
    if frames:
        output_path = os.path.join(OUTPUT_DIR, f"{lake_info['export_prefix']}_preview_60x120km.gif")
        create_animation(frames, output_path, fps=5)
        print(f"Preview saved: {output_path}")
    else:
        print("No frames available for preview")

# Create preview for first lake
create_preview(LAKES[0], num_days=60)

## Summary Statistics

In [ ]:
def generate_summary_stats(lake_info):
    """
    Generate summary statistics for mask status over time.
    """
    print(f"\nGenerating summary statistics for {lake_info['name']}...")
    
    stats = {'clear': 0, 'masked': 0, 'partial': 0}
    chl_stats = {'included': 0, 'excluded': 0}
    dates = []
    
    # Sample monthly to get statistics
    for year in range(2011, 2025):
        for month in range(1, 13):
            start = f"{year}-{month:02d}-01"
            if month == 12:
                end = f"{year+1}-01-01"
            else:
                end = f"{year}-{month+1:02d}-01"
            
            try:
                collection, roi, context = get_modis_collection(lake_info, start, end)
                images = collection.getInfo()['features']
                
                for img_info in images[:10]:  # Sample first 10 images per month
                    image = ee.Image(img_info['id'])
                    status = check_roi_mask_status(image, roi)
                    stats[status] += 1
                    
                    chl_excluded = check_chlorophyll_algorithm_mask(image, roi)
                    if chl_excluded:
                        chl_stats['excluded'] += 1
                    else:
                        chl_stats['included'] += 1
                    
                    dates.append(f"{year}-{month:02d}")
            except:
                continue
    
    # Calculate percentages
    total = sum(stats.values())
    if total > 0:
        print(f"\nMask Status Distribution for {lake_info['name']}:")
        print(f"  Clear:   {stats['clear']:4d} ({100*stats['clear']/total:.1f}%)")
        print(f"  Partial: {stats['partial']:4d} ({100*stats['partial']/total:.1f}%)")
        print(f"  Masked:  {stats['masked']:4d} ({100*stats['masked']/total:.1f}%)")
        print(f"  Total:   {total:4d} images sampled")
        
        print(f"\nChlorophyll Algorithm Status:")
        print(f"  Included: {chl_stats['included']:4d} ({100*chl_stats['included']/total:.1f}%)")
        print(f"  Excluded: {chl_stats['excluded']:4d} ({100*chl_stats['excluded']/total:.1f}%)")

# Generate statistics for each lake
for lake in LAKES:
    generate_summary_stats(lake)